# Crash Anticipation — Inference Timing Visualizations

This notebook reads `outputs/visualizations/timings.csv` and produces presentable visualizations to showcase model latency and throughput. It saves interactive HTML (and PNG if available) to `outputs/visualizations/figs/` for easy embedding on a website.


In [1]:
# Setup: import libraries (auto-install if missing)
import importlib, subprocess, sys

def ensure(pkg):
    try:
        return importlib.import_module(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        return importlib.import_module(pkg)

pd = ensure("pandas")
np = ensure("numpy")
plotly = ensure("plotly")
# Ensure nbformat for plotly rendering in notebooks
try:
    ensure("nbformat")
except Exception:
    pass
# Optional for static image export
try:
    ensure("kaleido")
except Exception:
    pass

px = importlib.import_module("plotly.express")
go = importlib.import_module("plotly.graph_objects")
from pathlib import Path



In [2]:
# Load timing data and compute derived metrics (robust path resolution)

def find_timing_csv() -> Path:
    candidates = [
        Path("outputs/visualizations/timings.csv"),            # repo root
        Path("../outputs/visualizations/timings.csv"),         # from demo/
        Path.cwd() / "outputs/visualizations/timings.csv",     # cwd-relative
        Path.cwd().parent / "outputs/visualizations/timings.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("Timing CSV not found in: " + ", ".join(str(p) for p in candidates))

csv_path = find_timing_csv()
print(f"Using CSV: {csv_path}")

df = pd.read_csv(csv_path)
# Derived metrics
df["effective_fps"] = df["num_frames"] / df["total_wall_s"].replace(0, np.nan)
df["model_fps"] = 1000.0 / df["avg_forward_ms"].replace(0, np.nan)
df["overhead_ms"] = df["total_wall_s"] * 1000.0 - df["total_forward_ms"]
df["id"] = df["id"].astype(str)

print("Rows:", len(df))
df.head(10)


Using CSV: ..\outputs\visualizations\timings.csv
Rows: 30


,id,num_frames,fps_input,num_inferences,total_forward_ms,avg_forward_ms,total_wall_s,output_path,effective_fps,model_fps,overhead_ms
0,593,50,10.0,35,736.141,21.033,1.780,outputs\visualizations\ccd_000593.mp4,28.089888,47.544335,1043.859
1,1257,50,10.0,35,613.192,17.520,1.657,outputs\visualizations\ccd_001257.mp4,30.175015,57.077626,1043.808
2,206,50,10.0,35,614.916,17.569,1.634,outputs\visualizations\ccd_000206.mp4,30.599755,56.918436,1019.084
3,493,50,10.0,35,631.839,18.053,1.735,outputs\visualizations\ccd_000493.mp4,28.818444,55.392456,1103.161
4,190,50,10.0,35,641.032,18.315,1.735,outputs\visualizations\ccd_000190.mp4,28.818444,54.600055,1093.968
5,1295,50,10.0,35,637.056,18.202,1.750,outputs\visualizations\ccd_001295.mp4,28.571429,54.939018,1112.944
6,1026,50,10.0,35,623.181,17.805,1.682,outputs\visualizations\ccd_001026.mp4,29.726516,56.163999,1058.819
7,649,50,10.0,35,631.605,18.046,1.687,outputs\visualizations\ccd_000649.mp4,29.638411,55.413942,1055.395
8,123,50,10.0,35,614.674,17.562,1.683,outputs\visualizations\ccd_000123.mp4,29.708853,56.941123,1068.326
9,1184,50,10.0,35,623.008,17.800,1.701,outputs\visualizations\ccd_001184.mp4,29.394474,56.179775,1077.992


In [3]:
# KPI summary
import numpy as np

def pct(s, q):
    return float(np.percentile(s.dropna(), q)) if len(s.dropna()) else float("nan")

metrics = {
    "avg_forward_ms": {
        "mean": df["avg_forward_ms"].mean(),
        "median": df["avg_forward_ms"].median(),
        "p95": pct(df["avg_forward_ms"], 95),
    },
    "effective_fps": {
        "mean": df["effective_fps"].mean(),
        "median": df["effective_fps"].median(),
        "p5": pct(df["effective_fps"], 5),
    },
}
metrics


{'avg_forward_ms': {'mean': np.float64(18.087533333333337),
  'median': np.float64(17.9795),
  'p95': 18.59855},
 'effective_fps': {'mean': np.float64(29.08785652567576),
  'median': np.float64(28.977305169396743),
  'p5': 27.95808976617553}}

In [4]:
# Bar: Effective throughput FPS per clip (sorted)
# Save figures next to the CSV under a `figs/` subfolder
out_dir = (csv_path.parent / "figs"); out_dir.mkdir(parents=True, exist_ok=True)

df_bar = df.sort_values("effective_fps", ascending=False)
fig = px.bar(
    df_bar,
    x="id",
    y="effective_fps",
    hover_data=["num_frames", "total_wall_s", "avg_forward_ms", "model_fps"],
    title="Effective Throughput (FPS) per Clip",
    labels={"id": "Clip ID", "effective_fps": "Effective FPS"},
)
fig.update_layout(template="plotly_white")
fig.show()

# Save
fig.write_html(out_dir / "effective_fps_bar.html")
try:
    fig.write_image(out_dir / "effective_fps_bar.png", scale=2)
except Exception as e:
    print("PNG export skipped (install 'kaleido' to enable).", e)



In [5]:
# Histogram + box of avg_forward_ms (model latency per inference)
fig = px.histogram(
    df,
    x="avg_forward_ms",
    nbins=30,
    marginal="box",
    title="Distribution of Model Latency (avg_forward_ms)",
    labels={"avg_forward_ms": "Avg Forward Pass (ms)"},
)
fig.update_layout(template="plotly_white")
fig.show()

fig.write_html(out_dir / "latency_hist.html")
try:
    fig.write_image(out_dir / "latency_hist.png", scale=2)
except Exception as e:
    pass



In [6]:
# Scatter: latency vs. num_frames, colored by effective_fps
fig = px.scatter(
    df,
    x="avg_forward_ms",
    y="num_frames",
    color="effective_fps",
    size="total_wall_s",
    hover_data=["id", "model_fps"],
    title="Latency vs. Clip Length",
    labels={"avg_forward_ms": "Avg Forward Pass (ms)", "num_frames": "Frames"},
)
fig.update_layout(template="plotly_white")
fig.show()

fig.write_html(out_dir / "latency_vs_frames.html")
try:
    fig.write_image(out_dir / "latency_vs_frames.png", scale=2)
except Exception:
    pass


In [7]:
# Stacked bars: forward vs overhead time per clip
stack_df = df.copy()
stack_df = stack_df.sort_values("total_wall_s", ascending=False)
fig = go.Figure()
fig.add_bar(name="Forward (ms)", x=stack_df["id"], y=stack_df["total_forward_ms"], marker_color="#1f77b4")
fig.add_bar(name="Overhead (ms)", x=stack_df["id"], y=stack_df["overhead_ms"], marker_color="#ff7f0e")
fig.update_layout(
    barmode="stack",
    title="Forward vs Overhead Time per Clip",
    xaxis_title="Clip ID",
    yaxis_title="Time (ms)",
    template="plotly_white",
)
fig.show()

fig.write_html(out_dir / "forward_vs_overhead.html")
try:
    fig.write_image(out_dir / "forward_vs_overhead.png", scale=2)
except Exception:
    pass


In [ ]:
# Save a lightweight summary CSV for the website (optional)
summary_cols = [
    "id",
    "num_frames",
    "fps_input",
    "effective_fps",
    "avg_forward_ms",
    "model_fps",
    "total_forward_ms",
    "overhead_ms",
    "total_wall_s",
    "output_path",
]
summary = df[summary_cols].sort_values(["effective_fps", "avg_forward_ms"], ascending=[False, True])
summary_path = out_dir / "timings_summary.csv"
summary.to_csv(summary_path, index=False)
summary.head(20)


,id,num_frames,fps_input,effective_fps,avg_forward_ms,model_fps,total_forward_ms,overhead_ms,total_wall_s,output_path
29,22,50,10.0,30.656039,17.167,58.251296,600.839,1030.161,1.631,outputs\visualizations\ccd_000022.mp4
2,206,50,10.0,30.599755,17.569,56.918436,614.916,1019.084,1.634,outputs\visualizations\ccd_000206.mp4
1,1257,50,10.0,30.175015,17.520,57.077626,613.192,1043.808,1.657,outputs\visualizations\ccd_001257.mp4
19,930,50,10.0,29.958059,17.465,57.257372,611.261,1057.739,1.669,outputs\visualizations\ccd_000930.mp4
6,1026,50,10.0,29.726516,17.805,56.163999,623.181,1058.819,1.682,outputs\visualizations\ccd_001026.mp4
8,123,50,10.0,29.708853,17.562,56.941123,614.674,1068.326,1.683,outputs\visualizations\ccd_000123.mp4
18,855,50,10.0,29.708853,17.658,56.631555,618.022,1064.978,1.683,outputs\visualizations\ccd_000855.mp4
7,649,50,10.0,29.638411,18.046,55.413942,631.605,1055.395,1.687,outputs\visualizations\ccd_000649.mp4
15,1483,50,10.0,29.585799,17.757,56.315819,621.485,1068.515,1.690,outputs\visualizations\ccd_001483.mp4
17,207,50,10.0,29.533373,17.899,55.869043,626.448,1066.552,1.693,outputs\visualizations\ccd_000207.mp4


: 

## Exported assets
- Interactive HTML plots: saved to `outputs/visualizations/figs/`
- Static PNGs (if `kaleido` installed): saved to the same folder
- CSV summary: `outputs/visualizations/figs/timings_summary.csv`

You can embed the HTML files on your website via an `<iframe>` or convert PNGs for static pages.
